In [1]:
import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers,models
import tensorflow_io as tfio
import os
import ast  # To safely convert the string "['bird1', 'bird2']" into a Python list

2026-04-11 06:47:40.525637: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775890060.781210      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775890060.850019      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775890061.411698      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775890061.411745      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775890061.411748      23 computation_placer.cc:177] computation placer alr

In [2]:
BASE_DIR = "/kaggle/input/notebooks/enoughrook2/birdclef-audio-decode"
final_labels_matrix = np.load(os.path.join(BASE_DIR, "final_labels.npy"))
raw_paths = np.load(os.path.join(BASE_DIR, "final_paths.npy"))
df = pd.read_csv('/kaggle/input/competitions/birdclef-2026/train.csv')
# The spectrograms themselves are inside the subfolder
# So when we re-map, we make sure they point to the subfolder correctly
final_npy_paths = np.array([p.replace('/kaggle/working/', BASE_DIR+ '/') for p in raw_paths])
print(f"Loaded {len(final_npy_paths)} file paths.")
print(f"Loaded {final_labels_matrix.shape} label matrix.")

Loaded 36939 file paths.
Loaded (36939, 206) label matrix.


In [3]:
def build_convnext_tiny(num_classes=206, input_shape=(128,313,3)):
    model= tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
    )
    model.trainable = True
    x= layers.GlobalAveragePooling2D()(model.output)
    
    x=layers.Dense(1024)(x)
    x=layers.BatchNormalization()(x)
    x=layers.Activation('gelu')(x)
    x=layers.Dropout(0.2)(x)

    head=layers.Dense(num_classes,activation='sigmoid')
    outputs=head(x)
    
    return models.Model(inputs=model.input, outputs=outputs)
    

def data_pipeline(path, label):
    # Load the .npy file
    img = tf.numpy_function(lambda p: np.load(p.decode('utf-8')), [path], tf.uint8)
    img = tf.cast(img, tf.float32)
    
    # DO NOT use set_shape with a fixed width here, as widths vary
    # Instead, we get the width of the current loaded file
    current_width = tf.shape(img)[1]
    img = img / 255.0

    # Apply Frequency Masking (Thunder/Noise removal)
    # This helps ignore low-frequency rumble below ~800Hz
    if tf.random.uniform([]) > 0.5:
        mask_height = 20
        # Create a mask that matches the DYNAMIC width of this specific file
        mask = tf.zeros([mask_height, current_width])
        img_bottom = img[mask_height:, :]
        img = tf.concat([mask, img_bottom], axis=0)
    
    # Prepare for ConvNeXt (add channel dimension and tile to 3 channels)
    img = tf.expand_dims(img, axis=-1)
    img = tf.tile(img, [1, 1, 3]) 
    
    # NOW we resize to the fixed shape the model expects
    # This standardizes the [128, 355] and [128, 438] files to [128, 313]
    img = tf.image.resize(img, [128, 313])
    
    return img, tf.cast(label, tf.float32)

In [4]:

mapping = {os.path.basename(f).split('.')[0]: label 
           for f, label in zip(df['filename'], df['primary_label'])}

valid_paths = []
valid_label_names = []


for p in final_npy_paths:
    # Extracts 'iNat870999' from '/path/22973_iNat870999.npy'
    fname = os.path.basename(p).replace('.npy', '')
    file_id = fname.split('_')[-1] 
    
    bird = mapping.get(file_id)
    if bird:
        valid_paths.append(p)
        valid_label_names.append([bird]) # MLB requires a list of lists

all_species = sorted(df['primary_label'].unique())
mlb = MultiLabelBinarizer(classes=all_species)

labels = mlb.fit_transform(valid_label_names).astype(np.float32)
paths = np.array(valid_paths)

print(f"Matched {len(valid_paths)} out of {len(final_npy_paths)} files.")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    paths, 
    labels, 
    test_size=0.2, 
    random_state=69
)

def create_dataset(paths, labels, is_training=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if is_training:
        ds = ds.shuffle(buffer_size=1000)
    
    ds = ds.map(data_pipeline, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Using (0.0, 0.0) ensures both image and label padding are float32
    ds = ds.padded_batch(
        16, 
        padded_shapes=([128, None, 3], [None]), 
        padding_values=(0.0, 0.0) 
    )
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = create_dataset(train_paths, train_labels, is_training=True)
val_ds = create_dataset(val_paths, val_labels, is_training=False)

print(f"Verified Alignment: {len(paths)} files matched.")
print(f"Training on: {len(train_paths)} samples")
print(f"Example Mapping: Path {train_paths[0]} matches Label Column {np.argmax(train_labels[0])}")

Matched 35461 out of 36939 files.


I0000 00:00:1775890088.170064      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Verified Alignment: 35461 files matched.
Training on: 28368 samples
Example Mapping: Path /kaggle/input/notebooks/enoughrook2/birdclef-audio-decode/processed_specs/rubthr1_XC419342.npy matches Label Column 142


In [5]:
model = build_convnext_tiny()
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-4, weight_decay=1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=[
        tf.keras.metrics.AUC(multi_label=True, name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)

111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [6]:

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'convnext_bird_model.keras',
        monitor='val_auc', # Focus on PR for bird detection
        save_best_only=True,
        mode='max',
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1,
        min_lr=1e-5 # prevents zero learining growth
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        patience=7,
        restore_best_weights=True,
        mode='max',
        min_delta=0.001,
    )
]

In [7]:
history = model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50


I0000 00:00:1775890117.757555      65 service.cc:152] XLA service 0x7d071c014c90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775890117.757603      65 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1775890122.609227      65 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-11 06:48:46.836427: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-11 06:48:47.012270: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-11 06:48:47.253753: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accur

1773/1773 ━━━━━━━━━━━━━━━━━━━━ 332s 158ms/step - auc: 0.4732 - loss: 0.3323 - precision: 0.0041 - recall: 0.0165 - val_auc: 0.4660 - val_loss: 0.2817 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 2.0000e-04
Epoch 2/50
1773/1773 ━━━━━━━━━━━━━━━━━━━━ 271s 153ms/step - auc: 0.4992 - loss: 0.1337 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_auc: 0.5275 - val_loss: 0.1533 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 2.0000e-04
Epoch 3/50
1773/1773 ━━━━━━━━━━━━━━━━━━━━ 270s 152ms/step - auc: 0.6233 - loss: 0.1325 - precision: 0.6698 - recall: 0.0021 - val_auc: 0.5265 - val_loss: 0.2239 - val_precision: 1.0000 - val_recall: 0.0037 - learning_rate: 2.0000e-04
Epoch 4/50
1773/1773 ━━━━━━━━━━━━━━━━━━━━ 273s 154ms/step - auc: 0.7764 - loss: 0.1298 - precision: 0.9215 - recall: 0.0876 - val_auc: 0.5789 - val_loss: 0.2182 - val_precision: 0.9580 - val_recall: 0.0193 - learning_rate: 2.0000e-04
Epoch 5/50
1773/1773 ━━━━━━━━━━━━━━━━━━━━ 272s 154m

In [8]:
print("Making predictions...")
y_true = np.concatenate([y for x, y in val_ds], axis=0)
val_preds = model.predict(val_ds)

best_f1 = 0
best_threshold = 0

for threshold in np.arange(0.05, 0.7, 0.02):
    preds_binary = (val_preds > threshold).astype(int)
    score = f1_score(y_true, preds_binary, average='macro', zero_division=0) #macro treats every label with equality
    
    print(f"Threshold: {threshold:.2f} | F1-Score: {score:.4f}")
    
    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print(f"\n Best F1-Score: {best_f1:.4f} at Threshold: {best_threshold}")

Making predictions...
444/444 ━━━━━━━━━━━━━━━━━━━━ 24s 45ms/step
Threshold: 0.05 | F1-Score: 0.0249
Threshold: 0.07 | F1-Score: 0.0699
Threshold: 0.09 | F1-Score: 0.1666
Threshold: 0.11 | F1-Score: 0.2673
Threshold: 0.13 | F1-Score: 0.3527
Threshold: 0.15 | F1-Score: 0.4164
Threshold: 0.17 | F1-Score: 0.4568
Threshold: 0.19 | F1-Score: 0.4882
Threshold: 0.21 | F1-Score: 0.5087
Threshold: 0.23 | F1-Score: 0.5265
Threshold: 0.25 | F1-Score: 0.5396
Threshold: 0.27 | F1-Score: 0.5456
Threshold: 0.29 | F1-Score: 0.5545
Threshold: 0.31 | F1-Score: 0.5609
Threshold: 0.33 | F1-Score: 0.5635
Threshold: 0.35 | F1-Score: 0.5639
Threshold: 0.37 | F1-Score: 0.5661
Threshold: 0.39 | F1-Score: 0.5665
Threshold: 0.41 | F1-Score: 0.5661
Threshold: 0.43 | F1-Score: 0.5662
Threshold: 0.45 | F1-Score: 0.5646
Threshold: 0.47 | F1-Score: 0.5641
Threshold: 0.49 | F1-Score: 0.5627
Threshold: 0.51 | F1-Score: 0.5623
Threshold: 0.53 | F1-Score: 0.5611
Threshold: 0.55 | F1-Score: 0.5590
Threshold: 0.57 | F1-Scor